# Tableau → Fabric: VizQL Data Service Bridge — Play 3

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This notebook is the **data loading** stage of the pipeline. It reads the datasource
and field inventory produced by Play 3 (Tableau Metadata Bridge), then uses the
VizQL Data Service (VDS) REST API to pull each upstream table from each datasource
into the Fabric Lakehouse as individual Delta tables.

**Pipeline order:** Play 2 → Play 3 → Play 4

```
Metadata_Lakehouse (Play 2 output)
  tableau_datasources  ← which datasources exist
  tableau_fields       ← which fields belong to which upstream table
        ↓
Play 3 (this notebook)
  For each datasource × upstream table:
    VDS query (fields for that table only → no joins)
        ↓
h1_ultrastore Lakehouse
  {datasource_name}_{table_name}  ← one Delta table per upstream table
        ↓
Play 4 → semantic model generation
```

**Delta table naming convention:** `{datasource_name}_{table_name}`
e.g. `superstore_datasource_orders`, `superstore_datasource_people`

---

**Prerequisites**
- Play 2 has been run and Metadata_Lakehouse tables are current
- Tableau Cloud or Tableau Server 2025.1+
- Creator license on the Tableau site
- Personal Access Token (PAT) stored in Azure Key Vault
- h1_ultrastore Lakehouse attached to this notebook

**Cells in this notebook**
1. Configuration
2. Authenticate to Tableau
3. Load Play 2 metadata
4. Main loop — query VDS per table and write Delta tables
5. Verification


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `PAT_NAME` | Tableau PAT name | Tableau → Account Settings → Personal Access Tokens |
| `POD` | Tableau Cloud pod hostname | First part of your Tableau Cloud URL |
| `SITE` | Site contentUrl slug | Your site URL slug. Use `""` for Tableau Server default site |
| `KV_URL` | Azure Key Vault URL | portal.azure.com → your Key Vault → Overview → Vault URI |
| `KV_SECRET_NAME` | PAT secret name in Key Vault | The secret name you used when storing the PAT |
| `METADATA_LAKEHOUSE` | Name of the Play 2 metadata lakehouse | e.g. `Metadata_Lakehouse` |
| `VDS_RATE_LIMIT` | Max VDS calls per hour | 100 × number of Creator licenses on your Tableau site |
| `DATASOURCE_FILTER` | Optional list of datasource names to process | Leave empty `[]` to process all |
| `BATCH_SIZE` | Max datasources per run | Use with `BATCH_OFFSET` to chunk large deployments |
| `BATCH_OFFSET` | Starting position in datasource list | Increment by `BATCH_SIZE` for next chunk |


## Cell 1 — Configuration

Set your Tableau environment details and pipeline controls here.
The PAT secret is retrieved securely from Azure Key Vault.

> 🔄 **Adapting for your environment:** Update `POD`, `SITE`, `KV_URL`, `KV_SECRET_NAME`,
> and `METADATA_LAKEHOUSE`. Adjust `VDS_RATE_LIMIT` based on your Creator license count.
> Use `DATASOURCE_FILTER` and batch controls for large deployments.

In [1]:
# ── TABLEAU CONNECTION ────────────────────────────────────────────────────────
PAT_NAME         = ""                                    # PAT name from Tableau account settings
POD              = ""                                    # e.g. 10ay.online.tableau.com
SITE             = ""                                    # Site contentUrl slug. Use "" for default site

KV_URL           = "https://<your-keyvault-name>.vault.azure.net/"
KV_SECRET_NAME   = "<your-secret-name>"

# ── LAKEHOUSE SETTINGS ───────────────────────────────────────────────────────
METADATA_LAKEHOUSE = "Metadata_Lakehouse"                # Play 2 output lakehouse name
# Note: h1_ultrastore (data lakehouse) must be attached as default lakehouse

# ── RATE LIMITING ────────────────────────────────────────────────────────────
VDS_RATE_LIMIT   = 100                                   # VDS calls/hour (100 × Creator license count)

# ── BATCH / FILTER CONTROLS ──────────────────────────────────────────────────
DATASOURCE_FILTER = []   # e.g. ["Superstore Datasource", "World Statistics"]
                         # Leave empty to process all datasources
PROJECT_FILTER    = []   # e.g. ["Finance", "Marketing"]
                         # Leave empty to process all projects
BATCH_SIZE        = 0    # Max datasources per run. 0 = no limit
BATCH_OFFSET      = 0    # Starting position. Increment by BATCH_SIZE for next chunk

# ── SECURE CREDENTIAL RETRIEVAL ──────────────────────────────────────────────
PAT_SECRET = notebookutils.credentials.getSecret(KV_URL, KV_SECRET_NAME)

BASE = f"https://{POD}"

import math
RATE_LIMIT_DELAY = 3600 / VDS_RATE_LIMIT   # seconds between VDS calls

print("✓ Configuration loaded")
print(f"  Pod:                {POD}")
print(f"  Site:               {SITE}")
print(f"  Metadata lakehouse: {METADATA_LAKEHOUSE}")
print(f"  VDS rate limit:     {VDS_RATE_LIMIT} calls/hour ({RATE_LIMIT_DELAY:.1f}s delay)")
print(f"  Datasource filter:  {DATASOURCE_FILTER or 'all'}")
print(f"  Project filter:     {PROJECT_FILTER or 'all'}")
print(f"  Batch:              offset={BATCH_OFFSET}, size={BATCH_SIZE or 'unlimited'}")
print(f"  PAT secret:         retrieved from Key Vault ✓")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 3, Finished, Available, Finished, False)

✓ Configuration loaded
  Pod:                10ay.online.tableau.com
  Site:               vdsapi82-5507b0d30c
  Metadata lakehouse: Metadata_Lakehouse
  VDS rate limit:     100 calls/hour (36.0s delay)
  Datasource filter:  all
  Project filter:     all
  Batch:              offset=0, size=unlimited
  PAT secret:         retrieved from Key Vault ✓


## Cell 2 — Authenticate to Tableau

Authenticates using your PAT and retrieves a session token.

> **If you get a 401 error in later cells**, re-run this cell to refresh the session token.
> Tokens expire after inactivity or if another session opens with the same PAT.

In [2]:
import requests
import json
import pandas as pd
import time
import re
from datetime import datetime

auth_response = requests.post(
    f"{BASE}/api/3.24/auth/signin",
    json={
        "credentials": {
            "personalAccessTokenName": PAT_NAME,
            "personalAccessTokenSecret": PAT_SECRET,
            "site": {"contentUrl": SITE}
        }
    },
    headers={"Content-Type": "application/json", "Accept": "application/json"}
)
auth_response.raise_for_status()

auth_data = auth_response.json()
TOKEN   = auth_data["credentials"]["token"]
SITE_ID = auth_data["credentials"]["site"]["id"]

HEADERS = {
    "X-Tableau-Auth": TOKEN,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

print("✓ Authenticated to Tableau")
print(f"  Token:    {TOKEN[:8]}...")
print(f"  Site ID:  {SITE_ID}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 4, Finished, Available, Finished, False)

✓ Authenticated to Tableau
  Token:    esSGFbHj...
  Site ID:  8c95a2ed-1221-46a6-ae25-1ff1a34e4c80


## Cell 3 — Load Play 3 Metadata

Reads the datasource and field inventory from the Metadata_Lakehouse tables
produced by Play 3. This is the manifest that drives the entire data loading loop.

Applies any configured filters (datasource name, project, batch offset/size)
so large deployments can be processed in chunks.

In [3]:
from pyspark.sql.types import NullType

def read_metadata_table(table_name):
    """Read a table from the Metadata Lakehouse, safely dropping void columns."""
    df = spark.sql(f"SELECT * FROM {METADATA_LAKEHOUSE}.dbo.{table_name}")
    void_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NullType)]
    if void_cols:
        df = df.drop(*void_cols)
    return df

# Load datasources inventory from Play 2
df_datasources = read_metadata_table("tableau_datasources").toPandas()

# Apply filters
if DATASOURCE_FILTER:
    df_datasources = df_datasources[df_datasources["name"].isin(DATASOURCE_FILTER)]
if PROJECT_FILTER:
    df_datasources = df_datasources[df_datasources["project_name"].isin(PROJECT_FILTER)]

# Apply batch offset/size
df_datasources = df_datasources.reset_index(drop=True)
if BATCH_SIZE > 0:
    df_datasources = df_datasources.iloc[BATCH_OFFSET:BATCH_OFFSET + BATCH_SIZE]
else:
    df_datasources = df_datasources.iloc[BATCH_OFFSET:]

# Load fields from Play 2 — ColumnFields with known source_table only
# These are the field captions passed to VDS queries
df_fields = read_metadata_table("tableau_fields").toPandas()
df_fields = df_fields[
    (df_fields["field_type"] == "ColumnField") &
    (df_fields["source_table"].notna())
]

print("✓ Metadata loaded from Play 2")
print(f"  Datasources to process: {len(df_datasources)}")
for _, ds in df_datasources.iterrows():
    tables = df_fields[df_fields["datasource_id"] == ds["datasource_id"]]["source_table"].unique()
    print(f"    • {ds['name']} → {list(tables)}")
print(f"  Total ColumnFields with source_table: {len(df_fields)}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 5, Finished, Available, Finished, False)

✓ Metadata loaded from Play 3
  Datasources to process: 1
    • Superstore Datasource → ['Orders', 'People', 'Returns']
  Total ColumnFields with source_table: 25


## Cell 4 — Main Loop: Reconcile Fields, Query VDS, Write Delta Tables

For each datasource, for each upstream table, this reconciles the **two Tableau APIs that
name the same physical column differently**, then lands one Delta table per source table:

1. **Payload columns** come from the VizQL Data Service `read-metadata` (the authoritative
   list of *queryable* field captions per logical table).
2. **Hidden join/grain keys** come from Play 2 metadata. VDS omits hidden fields (e.g. the
   `Row ID` grain key and hidden join keys like `Region (People)`), yet they ARE queryable
   by their disambiguated `<caption> (<table>)` name — so re-adding them preserves full row
   grain and keeps join keys for downstream relationships.
3. Each table's fields are queried **together** (one VDS call) so measures don't collapse and
   associations survive. On a 400, the offending field is isolated and dropped — never the batch.
4. Column names are sanitized for Delta, with deterministic `__2` suffixes on collisions.
5. Writes to `{datasource_name}_{table_name}` in the attached lakehouse.
6. Respects the VDS rate limit with a configurable delay between calls.

> **Manifests:** every field's fate is recorded — no silent drops. `tableau_landing_manifest`
> lists each caption (payload vs hidden key), its landed column, and any drop reason;
> `tableau_coverage_manifest` flags Metadata-API fields renamed/omitted by VDS.

> **On failure:** each table write is independent. If one fails, the loop continues and logs
> the error. Re-run with `DATASOURCE_FILTER` to retry specific datasources.

> **Rate limiting:** `VDS_RATE_LIMIT` in Cell 1 controls the delay between calls.
> Set to `100 × number of Creator licenses` on your Tableau site.

In [ ]:
# ── HELPERS ───────────────────────────────────────────────────────────────────
def clean_col(name):
    """Replace Delta-incompatible characters with underscores."""
    for ch in ["(", ")", " ", ",", ";", "{", "}", "/", "\\", "\n", "\t", "="]:
        name = name.replace(ch, "_")
    return name.strip("_")

def make_table_name(datasource_name, table_name):
    """Generate Delta table name from datasource and upstream table names."""
    def slugify(s):
        s = s.lower().strip()
        s = re.sub(r'[^a-z0-9]+', '_', s)
        return s.strip('_')
    return f"{slugify(datasource_name)}_{slugify(table_name)}"

def get_datasource_luid(datasource_name):
    """Look up the LUID for a datasource by name via REST API."""
    resp = requests.get(f"{BASE}/api/3.24/sites/{SITE_ID}/datasources", headers=HEADERS)
    resp.raise_for_status()
    datasources = resp.json().get("datasources", {}).get("datasource", [])
    if isinstance(datasources, dict):
        datasources = [datasources]
    match = next((ds for ds in datasources if ds["name"].lower() == datasource_name.lower()), None)
    if not match:
        raise ValueError(f"Datasource '{datasource_name}' not found on site")
    return match["id"]

def vds_read_metadata(luid):
    """Full VDS read-metadata: the authoritative list of QUERYABLE fields per logical table."""
    resp = requests.post(
        f"{BASE}/api/v1/vizql-data-service/read-metadata",
        json={"datasource": {"datasourceLuid": luid}}, headers=HEADERS)
    resp.raise_for_status()
    return resp.json().get("data", [])

def vds_query(luid, captions):
    """Query VDS for a list of field captions together (preserves row grain).
    Returns (rows, error_str). 429 raises to abort the datasource cleanly."""
    resp = requests.post(
        f"{BASE}/api/v1/vizql-data-service/query-datasource",
        json={"datasource": {"datasourceLuid": luid},
              "query": {"fields": [{"fieldCaption": c} for c in captions]},
              "options": {"returnFormat": "OBJECTS"}},
        headers=HEADERS)
    if resp.status_code == 200:
        return resp.json().get("data", []), None
    if resp.status_code == 429:
        raise RuntimeError("VDS rate limit (429) — re-run later or raise VDS_RATE_LIMIT")
    return None, f"{resp.status_code}: {resp.text[:200]}"

def write_delta(df_pandas, table_name):
    """Write pandas DataFrame to a Delta table in the attached (default) lakehouse."""
    df_spark = spark.createDataFrame(df_pandas)
    df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    return spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]

# ── CROSS-API FIELD RECONCILIATION ────────────────────────────────────────────
# The Metadata API and VizQL Data Service name the SAME physical column differently
# (e.g. Metadata "Person" vs VDS "Regional Manager"), and hidden join/grain keys
# (e.g. "Row ID", "Region (People)") are omitted from VDS read-metadata yet ARE
# queryable by the Metadata API's disambiguated "<caption> (<table>)" name. We therefore:
#   payload columns  <- VDS read-metadata (authoritative queryable names)
#   hidden keys      <- Metadata API hidden fields (join keys + the Row ID grain key)
# and query each table's fields TOGETHER so measures don't collapse and joins survive.

def _vds_table_of(logical_table_id):
    return logical_table_id.rsplit("_", 5)[0] if logical_table_id else ""

def _is_derived(field_name):
    f = (field_name or "").lower()
    return f.endswith("(bin)") or "(group)" in f or "(set)" in f

def payload_captions_by_table(vds_metadata):
    out = {}
    for v in vds_metadata:
        lt = _vds_table_of(v.get("logicalTableId", ""))
        if not lt or _is_derived(v.get("fieldName", "")):
            continue
        out.setdefault(lt, [])
        cap = v.get("fieldCaption")
        if cap and cap not in out[lt]:
            out[lt].append(cap)
    return out

def _as_bool(v):
    """Robust truthiness for an is_hidden value that may be bool, int, str, or NA."""
    if v is None:
        return False
    try:
        if pd.isna(v):
            return False
    except (TypeError, ValueError):
        pass
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return bool(v)

def hidden_captions_for_table(ds_fields, table_name):
    rows = ds_fields[ds_fields["source_table"] == table_name]
    return [r["field_name"] for _, r in rows.iterrows()
            if r.get("field_name") and _as_bool(r.get("is_hidden"))]

def _dedupe_collisions(captions):
    """Map captions -> landed Delta column names, suffixing clean_col collisions."""
    mapping, seen = {}, {}
    for cap in captions:
        base = clean_col(cap)
        if base not in seen:
            seen[base] = 1
            mapping[cap] = base
        else:
            seen[base] += 1
            mapping[cap] = f"{base}__{seen[base]}"
    return mapping

def query_table(luid, table_name, payload_caps, hidden_caps):
    """Query one table's payload + hidden keys together; isolate offenders on 400.
    Returns (renamed_dataframe, manifest_rows)."""
    candidates = [(c, "payload") for c in payload_caps]
    for c in hidden_caps:
        if c not in [x for x, _ in candidates]:
            candidates.append((c, "hidden_key"))
    caps = [c for c, _ in candidates]

    rows, err = vds_query(luid, caps)
    dropped = {}
    if rows is None and err and err.startswith("400"):
        print(f"    ⚠ Combined query 400 — isolating offending field(s)...")
        good = []
        for c in caps:
            time.sleep(RATE_LIMIT_DELAY)
            r, e = vds_query(luid, [c])
            if r is not None:
                good.append(c)
            else:
                dropped[c] = e
                print(f"    ✗ Not queryable in VDS, dropping: {c}")
        rows, err = (vds_query(luid, good) if good else ([], None))
        if rows is None:
            raise RuntimeError(f"Recombined VDS query failed for {table_name}: {err}")
        caps = good
    elif rows is None:
        raise RuntimeError(f"VDS query failed for {table_name}: {err}")

    landed = [c for c in caps if c not in dropped]
    colmap = _dedupe_collisions(landed)
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[[c for c in landed if c in df.columns]].rename(columns=colmap)

    # Transparency: a caption VDS accepted but did not return is recorded as dropped,
    # never silently omitted while the manifest claims it landed.
    returned = set(pd.DataFrame(rows).columns) if rows else set()
    missing = [c for c in landed if rows and c not in returned]
    for c in missing:
        dropped[c] = "accepted by VDS but absent from response"
    landed = [c for c in landed if c not in dropped]

    manifest = []
    for c, s in candidates:
        manifest.append({
            "datasource": None, "table": table_name, "caption": c, "source": s,
            "landed_column": colmap.get(c) if c not in dropped else None,
            "status": "dropped" if c in dropped else "landed",
            "reason": dropped.get(c, ""), "table_row_count": len(rows),
        })
    return df, manifest

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
results, errors, manifest_all, coverage_all = [], [], [], []
vds_call_count = 0

print(f"Starting VDS data load — {datetime.utcnow().isoformat()}")
print("=" * 60)

for _, ds_row in df_datasources.iterrows():
    ds_id, ds_name = ds_row["datasource_id"], ds_row["name"]
    print(f"\n── {ds_name} ──")
    try:
        luid = get_datasource_luid(ds_name)
        vds_meta = vds_read_metadata(luid)
        vds_call_count += 1
    except Exception as e:
        print(f"  ✗ Setup failed: {e}")
        errors.append({"datasource": ds_name, "table": "N/A", "error": str(e)})
        continue

    ds_fields = df_fields[df_fields["datasource_id"] == ds_id]
    payload_by_t = payload_captions_by_table(vds_meta)
    tables = sorted(set(payload_by_t) | set(ds_fields["source_table"].dropna().unique()))
    print(f"  Logical tables: {tables}")

    landed_by_table = {}
    for table_name in tables:
        delta_table = make_table_name(ds_name, table_name)
        payload_caps = payload_by_t.get(table_name, [])
        hidden_caps = hidden_captions_for_table(ds_fields, table_name)
        if not payload_caps and not hidden_caps:
            print(f"  → {table_name}: no queryable fields, skipping")
            continue
        if vds_call_count > 0:
            time.sleep(RATE_LIMIT_DELAY)
        try:
            df, manifest = query_table(luid, table_name, payload_caps, hidden_caps)
            vds_call_count += 1
            for m in manifest:
                m["datasource"] = ds_name
            manifest_all.extend(manifest)
            landed_by_table[table_name] = set(
                m["caption"] for m in manifest if m["status"] == "landed")
            if df.empty:
                print(f"  → {table_name}: 0 rows returned, skipping write")
                continue
            row_count = write_delta(df, delta_table)
            print(f"  ✓ {table_name} → {delta_table}: {row_count} rows, {len(df.columns)} cols {list(df.columns)}")
            results.append({"datasource": ds_name, "table": table_name,
                            "delta_table": delta_table, "rows": row_count, "cols": len(df.columns)})
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            errors.append({"datasource": ds_name, "table": table_name, "error": str(e)})

    # Coverage: Metadata-API ColumnFields that never landed (renamed/omitted by VDS)
    for _, f in ds_fields.iterrows():
        nm, t = f["field_name"], f["source_table"]
        landed = landed_by_table.get(t, set())
        if nm in landed:
            status = "landed"
        elif nm in payload_by_t.get(t, []):
            status = "vds_payload_renamed"
        else:
            status = "not_in_vds_metadata"
        coverage_all.append({"datasource": ds_name, "metadata_field": nm, "table": t,
                             "is_hidden": _as_bool(f.get("is_hidden")), "status": status})

# ── WRITE MANIFESTS (governance: every field's fate is recorded, no silent drops) ──
if manifest_all:
    mdf = pd.DataFrame(manifest_all)
    # Avoid Spark NullType on all-None object columns (e.g. an all-dropped run)
    for col in ("landed_column", "reason", "caption", "source", "table"):
        if col in mdf.columns:
            mdf[col] = mdf[col].astype(object).where(mdf[col].notna(), "")
    write_delta(mdf, "tableau_landing_manifest")
    print(f"\n  ✓ Landing manifest: tableau_landing_manifest ({len(manifest_all)} rows)")
if coverage_all:
    cdf = pd.DataFrame(coverage_all)
    for col in ("metadata_field", "table", "status"):
        if col in cdf.columns:
            cdf[col] = cdf[col].astype(object).where(cdf[col].notna(), "")
    write_delta(cdf, "tableau_coverage_manifest")
    print(f"  ✓ Coverage manifest: tableau_coverage_manifest ({len(coverage_all)} rows)")

print(f"\n{'=' * 60}")
print(f"✓ Complete — {datetime.utcnow().isoformat()}")
print(f"  VDS calls made: {vds_call_count}")
print(f"  Tables written: {len(results)}")
print(f"  Errors:         {len(errors)}")
dropped_rows = [m for m in manifest_all if m["status"] == "dropped"]
if dropped_rows:
    print(f"\n  Dropped fields (see tableau_landing_manifest):")
    for m in dropped_rows:
        print(f"    {m['datasource']} / {m['table']} / {m['caption']}: {m['reason']}")
if errors:
    print(f"\n  Failed tables:")
    for e in errors:
        print(f"    ✗ {e['datasource']} / {e.get('table')}: {e['error']}")


## Cell 5 — Verification

Confirms all expected Delta tables were written with the correct row counts.
Also shows the full list of tables written this run for handoff to Play 4.

In [5]:
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

if results:
    print("\n── Tables written this run ──")
    print(f"  {'Datasource':<30} {'Source Table':<20} {'Delta Table':<45} {'Rows'}")
    print(f"  {'-'*30} {'-'*20} {'-'*45} {'-'*8}")
    for r in results:
        print(f"  {r['datasource']:<30} {r['table']:<20} {r['delta_table']:<45} {r['rows']}")

    print(f"\n  Total tables: {len(results)}")
    print(f"  Total rows:   {sum(r['rows'] for r in results)}")
    print(f"\n  ✓ Delta tables ready for Play 4 semantic model generation")
    print(f"  ✓ Table naming: {{datasource_name}}_{{upstream_table}}")
else:
    print("  No tables written this run.")

if errors:
    print(f"\n── {len(errors)} error(s) — retry with DATASOURCE_FILTER ──")
    for e in errors:
        print(f"  ✗ {e['datasource']} / {e['table']}: {e['error']}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 7, Finished, Available, Finished, False)

VERIFICATION

── Tables written this run ──
  Datasource                     Source Table         Delta Table                                   Rows
  ------------------------------ -------------------- --------------------------------------------- --------
  Superstore Datasource          Orders               superstore_datasource_orders                  10194
  Superstore Datasource          People               superstore_datasource_people                  4
  Superstore Datasource          Returns              superstore_datasource_returns                 296

  Total tables: 3
  Total rows:   10494

  ✓ Delta tables ready for Play 4 semantic model generation
  ✓ Table naming: {datasource_name}_{upstream_table}
